# Method 2 — Full pipeline & đánh giá (Kaggle T4)

Phase 5 của `docs/method2_plan.md`: chạy oracle + full pipeline, xuất predictions theo contract, đo latency 4 giai đoạn. Ngân sách ~1h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 0: dò dataset + HF cache =====
# PHẢI chạy trước mọi import transformers: thư viện chốt cache lúc import,
# set HF_HOME sau đó thì không còn tác dụng.
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')


def _dirs_within(base: Path, max_depth: int = 4):
    """Mọi thư mục tới độ sâu `max_depth`, bỏ qua `hub/` cho nhanh."""
    frontier, seen = [base], []
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                children = [c for c in d.iterdir() if c.is_dir() and c.name != 'hub']
            except (PermissionError, OSError):
                continue
            seen.extend(children)
            nxt.extend(children)
        frontier = nxt
    return seen


def find_root(marker: str, label: str) -> Path:
    """Tìm thư mục chứa `marker`.

    Kaggle mount theo dạng /kaggle/input/datasets/<user>/<ds>/<ds>/, và số tầng
    đổi theo cách upload. Dò theo marker thì không phải hardcode username hay
    độ sâu — upload kiểu nào cũng tìm ra.
    """
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT):
        if (d / marker).exists():
            return d
    raise SystemExit(
        f'Không tìm thấy {label}: không thư mục nào dưới {INPUT_ROOT} có {marker}.\n'
        'Kiểm tra đã Add đủ 3 dataset ở sidebar Input chưa.'
    )


SRC_ROOT = find_root('src/models/preflight.py', 'dataset src')
DATA_ROOT = find_root('method2/manifest.json', 'dataset data')
HF_HOME = find_root('hub/models--BAAI--bge-m3', 'dataset hf-cache')

print('SRC :', SRC_ROOT)
print('DATA:', DATA_ROOT)
print('HF  :', HF_HOME)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# Có thư mục model chưa đủ — thiếu file trọng số thì lỗi chỉ lộ ra lúc nạp
# model, sau khi đã tốn thời gian cài đặt và copy.
for name in ('models--BAAI--bge-m3', 'models--xlm-roberta-base'):
    weights = [
        f for f in (HF_HOME / 'hub' / name).rglob('*')
        if f.is_file() and f.suffix in ('.safetensors', '.bin') and f.stat().st_size > 10**8
    ]
    assert weights, f'{name}: không có file trọng số > 100 MB'
    print(f'  {name}: {max(f.stat().st_size for f in weights) / 1024**3:.2f} GB')
print('\nHF cache OK')


In [ ]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# PEFT 0.20 raise nếu image có torchao < 0.16. Method 2 không dùng
# torchao quantization nên gỡ hẳn là xong.
!pip uninstall -y -q torchao 2>/dev/null || true

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
n_gpu = torch.cuda.device_count()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('số GPU:', n_gpu)

# sentence-transformers tự bọc DataParallel khi thấy >1 GPU. Với GradCache
# gọi model hàng trăm lần mỗi step thì phí đồng bộ cộng dồn rất nhanh.
if n_gpu > 1:
    print('  >1 GPU — truyền --single-gpu cho MỌI lệnh train')

# T4 là Turing (sm_75), KHÔNG có bf16 phần cứng. torch vẫn có thể báo
# is_bf16_supported()=True vì hỗ trợ qua emulation, chậm hơn fp16.
# Giữ fp16 bất kể giá trị này.
print('bf16 (emulated trên T4, vẫn dùng fp16):', torch.cuda.is_bf16_supported())


In [ ]:
# ===== Cell 2: copy code + data vào /kaggle/working =====
# Dataset chỉ đọc, mà code ghi checkpoint và dùng đường dẫn tương đối, nên
# phải copy sang thư mục ghi được. Dùng path đã dò ở Cell 0.
import shutil

WORK = Path('/kaggle/working')
# `scripts` cần thiết: benchmark_biencoder.py chạy trên Kaggle.
for name in ('src', 'configs', 'scripts'):
    target = WORK / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(SRC_ROOT / name, target)

# Dataset data bắt đầu thẳng bằng method2/ custom_vi/ benchmark_vi/ (KHÔNG có
# tầng `data/`), còn code tham chiếu `data/method2/...` → copy vào data/.
data_dir = WORK / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
data_dir.mkdir(parents=True)
for child in DATA_ROOT.iterdir():
    dest = data_dir / child.name
    shutil.copytree(child, dest) if child.is_dir() else shutil.copy2(child, dest)

%cd /kaggle/working

import json, glob, sys
sys.path.insert(0, '/kaggle/working')
# HF_HOME đã set ở Cell 0, kế thừa sang mọi tiến trình con `!python`.

print('src    :', sorted(p.name for p in (WORK / 'src').iterdir()))
print('data   :', sorted(p.name for p in data_dir.iterdir()))


In [ ]:
# ===== Cell 3: kiểm tra bản copy TRƯỚC khi preflight =====
# Preflight kiểm tra tính đúng đắn của dữ liệu; cell này kiểm tra bước copy —
# tách ra để khi hỏng thì biết ngay là hỏng ở đâu.
REQUIRED = [
    'data/method2/decontamination.json',
    'data/method2/manifest.json',
    'data/method2/tool_pool.json',
    'data/method2/biencoder/train.jsonl',
    'data/method2/biencoder/val.jsonl',
    'data/method2/biencoder/pairs_stats.json',
    'data/method2/crossencoder/train.jsonl',
    'data/method2/crossencoder/val.jsonl',
    'data/method2/label_stats.json',
    'data/custom_vi/v1/test_seen.jsonl',
    'data/benchmark_vi/test.jsonl',
    'configs/method2/biencoder.yaml',
    'configs/method2/crossencoder.yaml',
    'configs/method2/pinned_versions.json',
    'src/models/preflight.py',
]
missing = []
for rel in REQUIRED:
    path = WORK / rel
    if path.exists() and path.stat().st_size > 0:
        print(f'  {path.stat().st_size / 1024**2:8.2f} MB  {rel}')
    else:
        missing.append(rel)
        print(f'  {"THIẾU":>11}  {rel}')
assert not missing, f'Copy chưa đủ: {missing}'

# import được thì mới chắc src/ copy nguyên vẹn.
import importlib

importlib.import_module('src.models.preflight')
manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('\nsnapshot commit:', manifest.get('git_commit'))
print('copy OK')


## Pre-flight — cổng fail-closed TRƯỚC mọi training

```
decontamination.json tồn tại
        ↓
SHA-256 == manifest.json
        ↓
overlap train/val/test == 0
        ↓
unseen positive leakage == 0
        ↓
package versions khớp bản đã pin
        ↓
CHO PHÉP TRAIN
```

Thiếu file hoặc hash lệch → job dừng ngay, **không rebuild tự động**. Nếu
experiment chính tự dựng lại index từ dữ liệu đang có trên máy thì ta mất
đúng thứ cần đảm bảo: bằng chứng model được train trên đúng split đã kiểm
định. Rebuild là lệnh preprocessing riêng, chạy ở local rồi upload lại:
`python -m src.models.sources decontaminate && python -m src.models.sources manifest`

Vì sao `val ∩ test` là rủi ro nặng nhất: dù không train trên query đó, việc
chọn checkpoint/hyperparameter bằng val vẫn khiến metric test lạc quan hơn
thực tế. `data/benchmark_vi` **giữ nguyên** — decontamination nằm ở tầng
dataset của Method 2 nên bốn method vẫn được đánh giá trên cùng một tập test.


In [ ]:
# Exit code != 0 → dừng notebook, không chạy tiếp cell training nào.
!python -m src.models.preflight \
    --config configs/method2/biencoder.yaml \
    --require-gpu T4 \
    --output results/method2/preflight.json

preflight = json.load(open('results/method2/preflight.json', encoding='utf-8'))
assert preflight['passed'], f"Preflight KHÔNG ĐẠT: {preflight['failures']}"
print('preflight PASS —', len(preflight['checks']), 'check')


In [ ]:
# Số liệu split để đối chiếu bằng mắt trước khi tiêu giờ GPU.
stats = json.load(open('data/method2/biencoder/pairs_stats.json', encoding='utf-8'))
decon = stats['decontamination']

print('unique query/split :', stats['unique_queries_per_split'])
print('positive pairs     :', stats['n_positive_pairs'])
print('negative samples   :', stats['n_negative_samples'])
print('query trùng split  :', decon['n_overlapping_queries'], decon['overlapping_queries'])
print('sample bị loại     :', decon['rows_dropped_total'], decon['rows_dropped_by_transition'])
print('overlap còn lại    :', stats['split_overlap_after'])


# Phase 5 — Full pipeline & đánh giá

Bi-Encoder → Cross-Encoder → Validator, xuất `predictions.jsonl` theo
prediction contract rồi chấm bằng evaluator chung `src/evaluation`.

**Hai chế độ bắt buộc** (§6.3 `experimental_plan.md`):

| Chế độ | Nghĩa |
|---|---|
| `oracle` | đưa thẳng tool đúng vào Cross-Encoder → đo riêng extraction |
| `pipeline` | Bi-Encoder retrieve rồi Cross-Encoder extract → đo cả chuỗi |

Chênh lệch giữa hai chế độ cho biết lỗi nằm ở retrieval hay extraction.

**Ngưỡng τ đã FREEZE từ val** (`run02/thresholds.json`). Đây là lần chạy
trên **test** — tune lại ngưỡng ở đây là làm hỏng toàn bộ kết quả.

Cần 5 dataset: `src`, `data`, `hf-cache`, **`biencoder-run02`**,
**`crossencoder-run01`** (mỗi checkpoint là thư mục `final/`).


## Nạp hai checkpoint từ Kaggle Dataset

`/kaggle/working` không sống qua session nên cả hai model phải được
upload lại. Dò theo marker để không hardcode tên dataset — Bi-Encoder có
`modules.json` (sentence-transformers), Cross-Encoder có
`crossencoder_heads.pt`.


In [ ]:
import re, shutil, subprocess, sys

BI = '/kaggle/working/artifacts/method2/biencoder/run02'
CE = '/kaggle/working/artifacts/method2/crossencoder/run01'


def find_optional(markers, max_depth=6):
    """Thư mục chứa TẤT CẢ marker. Nhiều marker là cố ý:

    `final/modules.json` khớp cả run01 lẫn run02 của Bi-Encoder — hai thư mục
    giống hệt nhau về cấu trúc. Nếu dataset run01 cũ còn được Add vào notebook
    này, nó có thể được chọn trước và toàn bộ đánh giá cuối chạy bằng
    checkpoint sai mà không có dấu hiệu gì. `thresholds.json` chỉ đi kèm run02
    (hiệu chỉnh sau Round 2) nên nó phân biệt được hai bên.
    """
    if isinstance(markers, str):
        markers = [markers]
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT, max_depth):
        if all((d / m).exists() for m in markers):
            return d
    return None


for dest, markers, label in (
    (BI, ['final/modules.json', 'thresholds.json'], 'Bi-Encoder run02'),
    (CE, ['final/crossencoder_heads.pt'], 'Cross-Encoder run01'),
):
    if Path(f'{dest}/final').exists():
        print(f'{label}: đã có sẵn')
        continue
    found = find_optional(markers)
    assert found, (
        f'THIẾU dataset {label}: không thư mục nào có đủ {markers}. '
        'Với Bi-Encoder, thresholds.json PHẢI nằm cùng cấp với final/ — '
        'đó cũng là thứ phân biệt run02 với run01.'
    )
    shutil.copytree(found / 'final', f'{dest}/final', dirs_exist_ok=True)
    print(f'{label}: nạp từ {found}')
    if 'thresholds.json' in markers:
        shutil.copy2(found / 'thresholds.json', f'{dest}/thresholds.json')

# Ngưỡng hiệu chỉnh trên val rồi FREEZE. Tự sửa nếu `strategy` lệch với
# `winner` đã tính sẵn trong chính file — bug từng gặp: config cũ hardcode
# strategy="absolute" nên file luôn ghi "absolute" dù calibration tự chọn
# "gap" tốt hơn. gap_delta đã tính đúng sẵn nên KHÔNG cần GPU để sửa — chỉ
# sửa bản copy trong /kaggle/working, không đụng tới Kaggle Dataset gốc.
from src.models.biencoder.evaluate import reconcile_strategy

raw_thr = json.load(open(f'{BI}/thresholds.json', encoding='utf-8'))
thr, _fixed = reconcile_strategy(raw_thr)
if _fixed:
    print(f'SỬA strategy: {raw_thr["strategy"]!r} -> {thr["strategy"]!r}',
          '(khớp winner đã tính sẵn trong calibration, không cần GPU)')
    json.dump(thr, open(f'{BI}/thresholds.json', 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2)
print(f"tau={thr['tau']} tau_call={thr['tau_call']} gap_delta={thr['gap_delta']}",
      f"strategy={thr['strategy']} k_max={thr['k_max']}",
      f"hiệu chỉnh trên {thr['calibrated_on']}")


## Dựng lại index tool

`data/method2/index/` được sinh trong session Bi-Encoder và **không** nằm
trong dataset upload. Dựng lại mất ~50 s cho 4,464 tool — rẻ hơn nhiều so
với upload 18 MB embedding, và bảo đảm index khớp đúng checkpoint đang dùng.

`t_index_build` ghi riêng vào `index_meta.json`, **không** cộng vào latency
mỗi query (§10.1 `experimental_plan.md`) — đây là chi phí một lần.


In [ ]:
if not Path('data/method2/index/tool_embeddings.npy').exists():
    subprocess.run(
        [sys.executable, '-m', 'src.models.biencoder.index',
         '--config', 'configs/method2/biencoder.yaml', '--model', f'{BI}/final'],
        check=True,
    )
else:
    print('index đã có — bỏ qua')

meta = json.load(open('data/method2/index/index_meta.json', encoding='utf-8'))
print('index:', meta['n_tools'], 'tool,', meta['t_index_build_sec'], 's')


## Chạy pipeline + oracle trên 3 tập test

`custom_seen` · `custom_unseen` · `benchmark`, mỗi tập 2 chế độ = 6 lần
chạy. Ước tính ~1 h.


In [ ]:
# subprocess thay vì `!` trong vòng lặp: exit code hiện ra rõ ràng nên một
# lần chạy hỏng không bị trôi qua trong Save & Run All.
GOLD = [
    ('data/custom_vi/v1/test_seen.jsonl', 'custom_seen'),
    ('data/custom_vi/v1/test_unseen.jsonl', 'custom_unseen'),
    ('data/benchmark_vi/test.jsonl', 'benchmark'),
]

for gold, tag in GOLD:
    for mode in ['pipeline', 'oracle']:
        out = f'results/method2/predictions/{tag}'
        done = f'{out}/' + ('predictions.jsonl' if mode == 'pipeline'
                            else 'oracle_predictions.jsonl')
        if Path(done).exists():
            print(f'=== {tag} / {mode}: đã có, bỏ qua ===', flush=True)
            continue
        print(f'=== {tag} / {mode} ===', flush=True)
        subprocess.run(
            [sys.executable, '-m', 'src.models.pipeline.method2',
             '--config', 'configs/method2/pipeline.yaml',
             '--gold', gold, '--mode', mode, '--output-dir', out],
            check=True,
        )


## Evaluator chung — cả 3 tập

`--slice metadata.tool_split` tách seen/unseen, đúng như Bảng D §11
`experimental_plan.md` yêu cầu. `--oracle-predictions` cho phép evaluator
tính chênh lệch pipeline vs oracle trong cùng một báo cáo.


In [ ]:
for gold, tag in GOLD:
    print(f'=== evaluate {tag} ===', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'src.evaluation.cli', 'evaluate',
         '--gold', gold,
         '--predictions', f'results/method2/predictions/{tag}/predictions.jsonl',
         '--oracle-predictions',
         f'results/method2/predictions/{tag}/oracle_predictions.jsonl',
         '--slice', 'metadata.tool_split',
         '--output-dir', f'results/evaluation/method_2_{tag}'],
        check=True,
    )


## Chan doan: chien luoc chon tool

`raw_predictions_pipeline.jsonl` da luu `ranked_tools` kem score, nen doi
nguong hay doi chien luoc la **tinh lai duoc offline** - khong can GPU,
khong can chay lai pipeline.

Cot `#tool chon TB` la thu can nhin: CustomTools-VI co hard negative cung
`feature_group` nen diem sat nhau. Neu `absolute` chon trung binh 2-3 tool
trong khi gold chi 1, do chinh la cho Tool Set Accuracy roi.

Nguong chinh thuc VAN phai freeze tu val (§5.1) - bang nay chi de chan doan,
khong duoc chon nguong bang cach quet tren test.


In [ ]:
for gold, tag in GOLD:
    raw_path = f'results/method2/predictions/{tag}/raw_predictions_pipeline.jsonl'
    if not Path(raw_path).exists():
        print(f'{tag}: chua co {raw_path}')
        continue
    print(f'=== {tag} ===')
    subprocess.run(
        [sys.executable, 'scripts/method2/replay_selection.py',
         '--gold', gold, '--raw', raw_path,
         '--thresholds', f'{BI}/thresholds.json', '--sweep'],
        check=True,
    )
    print()


## Bảng kết quả — Bảng C và D §11

Evaluator ghi `report.json`, `per_sample.jsonl`, `summary.md` vào
`results/evaluation/method_2_<tag>/`. Cell dưới đọc ra ba thứ:

1. **Bảng tổng** — metric chính của cả 3 tập, cạnh nhau.
2. **Chênh lệch oracle vs pipeline** — `oracle_arg_em − pipeline_arg_em`.
   Chênh nhỏ nghĩa là lỗi nằm ở extraction; chênh lớn nghĩa là retrieval
   đang kéo cả chuỗi xuống (§6.3 `experimental_plan.md`).
3. **Theo `tool_split`** — seen vs unseen, đúng Bảng D.

Cell này chỉ đọc file, không tốn GPU.


In [ ]:
REPORTS = {}
for _, tag in GOLD:
    path = Path(f'results/evaluation/method_2_{tag}/report.json')
    if path.exists():
        REPORTS[tag] = json.load(open(path, encoding='utf-8'))
    else:
        print(f'THIẾU {path}')

ROWS = [
    ('Call F1',              lambda m: m['detection']['f1']),
    ('Recall@5',             lambda m: m['retrieval'].get('recall_at_5')),
    ('Tool Set Accuracy',    lambda m: m['selection']['tool_set_accuracy_positive']),
    ('ArgEM | đúng tool',    lambda m: m['extraction']['normalized_arg_em_given_correct_tool']),
    ('ArgEM | ORACLE tool',  lambda m: (m['oracle_extraction'] or {}).get('normalized_arg_em_given_correct_tool')),
    ('Argument Pair F1',     lambda m: m['extraction']['argument_pair']['f1']),
    ('Schema Validity',      lambda m: m['schema_validity']['call_schema_validity']),
    ('N-FCEM positive',      lambda m: m['end_to_end']['n_fcem_positive']),
    ('Overall Success',      lambda m: m['end_to_end']['overall_success']),
]


def fmt(v):
    return '  —   ' if v is None else f'{v:6.4f}'


tags = list(REPORTS)
print(f"{'metric':22s}" + ''.join(f'{t:>16s}' for t in tags))
print('-' * (22 + 16 * len(tags)))
for name, get in ROWS:
    cells = ''.join(f'{fmt(get(REPORTS[t]["metrics"])):>16s}' for t in tags)
    print(f'{name:22s}' + cells)

print()
print('Chênh lệch oracle - pipeline (dương = retrieval đang kéo xuống):')
for t in tags:
    m = REPORTS[t]['metrics']
    pipe = m['extraction']['normalized_arg_em_given_correct_tool']
    orac = (m['oracle_extraction'] or {}).get('normalized_arg_em_given_correct_tool')
    gap = None if (pipe is None or orac is None) else orac - pipe
    print(f'  {t:16s} pipeline={fmt(pipe)}  oracle={fmt(orac)}  chênh={fmt(gap)}')


### Theo `tool_split` — Bảng D

`robustness` gom theo từng slice field. `metadata.tool_split` chỉ có ở
custom_vi (benchmark không gắn nhãn seen/unseen).


In [ ]:
for t in tags:
    rob = REPORTS[t]['metrics'].get('robustness') or {}
    groups = (rob.get('metadata.tool_split') or {}).get('groups')
    if not groups:
        print(f'{t}: không có slice metadata.tool_split')
        continue
    print(f'--- {t} ---')
    hdr = f"{'split':10s} {'n':>6s} {'overall':>9s} {'N-FCEM':>9s} {'toolset':>9s} {'ArgEM':>9s}"
    print(hdr)
    for split, g in groups.items():
        print(f"{split:10s} {g['sample_count']:6d} {fmt(g['overall_success']):>9s}",
              f"{fmt(g['n_fcem_positive']):>9s} {fmt(g['tool_set_accuracy_positive']):>9s}",
              f"{fmt(g['normalized_arg_em_given_correct_tool']):>9s}")
    print()


## Kết quả — Bảng C và Bảng D (§11 `experimental_plan.md`)

Evaluator ghi `report.json` + `summary.md` + `per_sample.jsonl` cho mỗi
tập nhưng không in gì ra. Không đọc lại thì cả Phase 5 chỉ tạo file mà
không ai nhìn thấy con số nào.

**Chênh lệch ArgEM giữa pipeline và oracle** là số quan trọng nhất: nó
tách lỗi retrieval khỏi lỗi extraction (§6.3).


In [ ]:
import collections

REPORTS = {}
for _, tag in GOLD:
    path = Path(f'results/evaluation/method_2_{tag}/report.json')
    if not path.exists():
        print(f'{tag}: THIẾU {path}')
        continue
    REPORTS[tag] = json.load(open(path, encoding='utf-8'))


def _f(value):
    return '     —' if value is None else f'{value:6.4f}'


hdr = (f"{'tập':15s} {'n':>6s} {'CallF1':>7s} {'ToolSet':>7s} {'ArgEM':>7s}"
       f" {'oracle':>7s} {'Δ':>7s} {'Schema':>7s} {'N-FCEM':>7s} {'Success':>7s}")
print(hdr)
print('-' * len(hdr))
for tag, rep in REPORTS.items():
    m = rep['metrics']
    arg = m['extraction']['normalized_arg_em_given_correct_tool']
    orc = (m['oracle_extraction'] or {}).get('normalized_arg_em_given_correct_tool')
    gap = None if (arg is None or orc is None) else orc - arg
    print(f"{tag:15s} {rep['dataset']['sample_count']:6d}",
          _f(m['detection']['f1']),
          _f(m['selection']['tool_set_accuracy_positive']),
          _f(arg), _f(orc), _f(gap),
          _f(m['schema_validity']['call_schema_validity']),
          _f(m['end_to_end']['n_fcem_positive']),
          _f(m['end_to_end']['overall_success']))

# Bảng D — seen vs unseen. `--slice metadata.tool_split` đi vào `robustness`.
print()
print('theo tool_split (Bảng D):')
for tag, rep in REPORTS.items():
    field = (rep['metrics'].get('robustness') or {}).get('metadata.tool_split')
    for value, g in ((field or {}).get('groups') or {}).items():
        print(f"  {tag:15s} {value:8s} n={g['sample_count']:5d}",
              'ToolSet=' + _f(g['tool_set_accuracy_positive']),
              'ArgEM=' + _f(g['normalized_arg_em_given_correct_tool']),
              'Success=' + _f(g['overall_success']))

# Phân loại lỗi W/T/P/I (§9). Đếm trên từng PARAMETER, không phải từng
# sample — nên tổng lớn hơn số sample là bình thường.
print()
print('phân loại lỗi (§9):')
for _, tag in GOLD:
    path = Path(f'results/method2/predictions/{tag}/errors_pipeline.jsonl')
    if not path.exists():
        continue
    counts = collections.Counter(
        json.loads(line)['error_class'] for line in open(path, encoding='utf-8')
    )
    print(f'  {tag:15s}', dict(counts.most_common()))


## Latency — tách 4 giai đoạn

§10.1 `experimental_plan.md` bắt buộc tách riêng thời gian embed query,
retrieve, cross-encode và validate. `t_index_build` là chi phí một lần,
báo cáo riêng, **không** cộng vào latency mỗi query.


In [ ]:
for _, tag in GOLD:
    path = f'results/method2/predictions/{tag}/latency_pipeline.json'
    if not Path(path).exists():
        print(f'{tag}: chưa có {path}')
        continue
    latency = json.load(open(path, encoding='utf-8'))
    print(f'--- {tag} ---')
    for stage in ['t_query_embed', 't_retrieve', 't_cross_encode',
                  't_validate', 'total']:
        if stage in latency:
            print(f"  {stage:16s} p50={latency[stage]['p50_ms']:8.2f} ms",
                  f"p95={latency[stage]['p95_ms']:8.2f} ms")

print()
print('t_index_build (một lần, KHÔNG cộng vào latency/query):',
      meta['t_index_build_sec'], 's')


## Run manifest — audit §11 `method2_plan.md`

Mỗi run phải tự chứng minh được là chạy lại duoc: commit, hash config,
fingerprint dataset, seed, artifact. Hai notebook train đã sinh manifest;
notebook này trước đây thì không, nên Phase 5 thiếu đúng mục đó.

`--stage evaluation` dùng bộ check riêng: run đánh giá không có
checkpoint/VRAM/duration mà có predictions + latency + errors + seed.
Dùng bộ check của train ở đây sẽ báo thiếu giả.


In [ ]:
for _, tag in GOLD:
    out = f'results/method2/predictions/{tag}'
    report = f'results/evaluation/method_2_{tag}/report.json'
    if not Path(report).exists():
        print(f'{tag}: chua co {report} — bo qua')
        continue
    subprocess.run(
        [sys.executable, '-m', 'src.models.run_manifest',
         '--run-dir', out, '--config', 'configs/method2/pipeline.yaml',
         '--stage', 'evaluation', '--report', f'metrics={report}'],
        check=True,
    )
    man = json.load(open(f'{out}/run_manifest.json', encoding='utf-8'))
    print(f"{tag:14s} commit={man['git']['commit'][:12] if man['git']['commit'] else None}",
          f"seed={man['evaluation']['seed']}",
          f"thiếu: {man['audit_complete']['missing'] or 'không'}")


## Ablation §6.1 — `should_call` head thay ngưỡng cosine τ

§7.10e cho thấy abstention là thứ chặn trần Overall Success, nên đây là
ablation có kỳ vọng cao nhất trong Phase 6. Chỉ chạy được khi checkpoint
Cross-Encoder được train với `model.enable_should_call: true`; checkpoint
baseline không có head đó và pipeline sẽ **dừng** chứ không lặng lẽ quay
về τ — quay lặng lẽ là báo cáo số baseline dưới tên ablation.

Thứ tự bắt buộc, giống hệt τ: chạy **val** → chốt ngưỡng → FREEZE → mới
chạy test. Ngưỡng được dò OFFLINE từ `metadata.should_call_prob` mà
pipeline đã ghi, nên quét lại bao nhiêu lần cũng không tốn thêm GPU.


In [ ]:
# Đặt True khi đang chạy nhánh ablation §6.1 (checkpoint có head should_call).
RUN_SHOULD_CALL = False

SC_VAL = [
    ('data/custom_vi/v1/val_seen.jsonl', 'val_seen'),
    ('data/custom_vi/v1/val_unseen.jsonl', 'val_unseen'),
]
SC_THRESHOLD_PATH = 'artifacts/method2/crossencoder/should_call_threshold.json'
SC_CHECKPOINT = '/kaggle/working/artifacts/method2/ablation/should_call'


def _has_should_call_head(directory):
    """Checkpoint này có head should_call không — đọc NỘI DUNG, không đoán tên.

    Cả run01 lẫn nhánh ablation đều có đúng file `final/crossencoder_heads.pt`,
    nên `find_optional` bốc trúng cái nào là tuỳ thứ tự duyệt thư mục. Đó là
    đúng cái bẫy mà docstring của `find_optional` cảnh báo cho run01/run02,
    lần này ở phía Cross-Encoder. `head_config` nằm sẵn trong checkpoint và
    trả lời dứt khoát, nên hỏi nó.
    """
    import torch

    path = Path(directory) / 'final' / 'crossencoder_heads.pt'
    if not path.exists():
        return False
    try:
        config = torch.load(path, map_location='cpu').get('head_config') or {}
    except Exception:
        return False
    return bool(config.get('enable_should_call'))


if RUN_SHOULD_CALL:
    assert not _has_should_call_head(CE), (
        'Checkpoint Phase 5 lại là bản CÓ head should_call — nghĩa là hai dataset '
        'checkpoint đã bị bốc lẫn, và bảng Phase 5 ở trên không còn là baseline.'
    )
    if not _has_should_call_head(SC_CHECKPOINT):
        found = next(
            (d for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT, 6)
             if _has_should_call_head(d)),
            None,
        )
        assert found, (
            'THIẾU dataset checkpoint nhánh should_call: không thư mục input nào '
            'có final/crossencoder_heads.pt với enable_should_call=true.'
        )
        shutil.copytree(found / 'final', f'{SC_CHECKPOINT}/final', dirs_exist_ok=True)
        print('should_call: nạp từ', found)
    print('Phase 5 dùng', CE, '| ablation dùng', SC_CHECKPOINT)


In [ ]:
if RUN_SHOULD_CALL:
    # Vòng 1 — VAL. Ngưỡng 0.5 chỉ để pipeline chạy được và ghi ra prob;
    # con số dùng để báo cáo đến từ bước hiệu chỉnh ngay dưới.
    for gold, tag in SC_VAL:
        out = f'results/method2/should_call/{tag}'
        if Path(f'{out}/predictions.jsonl').exists():
            print(f'=== {tag}: đã có, bỏ qua ==='); continue
        print(f'=== should_call / {tag} ===', flush=True)
        subprocess.run(
            [sys.executable, '-m', 'src.models.pipeline.method2',
             '--config', 'configs/method2/pipeline.yaml',
             '--gold', gold, '--mode', 'pipeline', '--output-dir', out,
             '--crossencoder', f'{SC_CHECKPOINT}/final',
             '--abstention', 'should_call'],
            check=True,
        )
else:
    print('RUN_SHOULD_CALL=False — bỏ qua ablation §6.1')


In [ ]:
if RUN_SHOULD_CALL:
    # Gộp hai tập val thành một để dò ngưỡng: val_unseen chỉ có 865 sample
    # no-call ở toàn bộ split val, chia nhỏ nữa thì ngưỡng chọn ra là nhiễu.
    merged_gold = 'results/method2/should_call/val_all_gold.jsonl'
    merged_pred = 'results/method2/should_call/val_all_pred.jsonl'
    for dst, parts in ((merged_gold, [g for g, _ in SC_VAL]),
                       (merged_pred, [f'results/method2/should_call/{t}/predictions.jsonl'
                                      for _, t in SC_VAL])):
        with open(dst, 'w', encoding='utf-8') as out_file:
            for part in parts:
                out_file.write(Path(part).read_text(encoding='utf-8'))
    subprocess.run(
        [sys.executable, 'scripts/method2/calibrate_should_call.py',
         '--gold', merged_gold, '--predictions', merged_pred,
         '--output', SC_THRESHOLD_PATH],
        check=True,
    )
    frozen = json.load(open(SC_THRESHOLD_PATH, encoding='utf-8'))
    print('ngưỡng đã FREEZE:', frozen['should_call_threshold'],
          '| macro_f1 val:', frozen['metrics']['macro_f1'])


In [ ]:
if RUN_SHOULD_CALL:
    # Vòng 2 — TEST, bằng ngưỡng vừa freeze. Không dò lại ở đây.
    sc_tau = json.load(open(SC_THRESHOLD_PATH, encoding='utf-8'))['should_call_threshold']

    # Chạy checkpoint NÀY dưới CẢ HAI cơ chế. So thẳng với bảng Phase 5 là
    # so nhầm: đó là run01 — checkpoint khác, train trên bộ pair khác (+22,8%
    # dòng). Chênh lệch khi ấy gộp cả 'đổi cơ chế' lẫn 'đổi checkpoint', và
    # không tách ra được nữa. Thêm một lượt pipeline (~7 phút/tập) là giá rẻ
    # cho một phép so sạch.
    SC_MODES = [
        ('tau', ['--abstention', 'tau']),
        ('should_call', ['--abstention', 'should_call',
                         '--should-call-threshold', str(sc_tau)]),
    ]

    for gold, tag in GOLD:
        for label, flags in SC_MODES:
            out = f'results/method2/should_call/{label}/{tag}'
            if Path(f'{out}/predictions.jsonl').exists():
                print(f'=== {tag} / {label}: đã có, bỏ qua ==='); continue
            print(f'=== {tag} / {label} ===', flush=True)
            subprocess.run(
                [sys.executable, '-m', 'src.models.pipeline.method2',
                 '--config', 'configs/method2/pipeline.yaml',
                 '--gold', gold, '--mode', 'pipeline', '--output-dir', out,
                 '--crossencoder', f'{SC_CHECKPOINT}/final']
                + flags,
                check=True,
            )
            subprocess.run(
                [sys.executable, '-m', 'src.evaluation.cli', 'evaluate',
                 '--gold', gold, '--predictions', f'{out}/predictions.jsonl',
                 '--slice', 'metadata.tool_split',
                 '--output-dir', f'results/evaluation/sc_{label}_{tag}'],
                check=True,
            )

    # Cùng tập, cùng checkpoint, chỉ khác cơ chế abstention.
    print()
    print(f"{'tập':15s} {'cơ chế':12s} {'NegRecall':>10s} {'CallF1':>8s} {'Success':>8s}")
    for _, tag in GOLD:
        for label, _ in SC_MODES:
            path = Path(f'results/evaluation/sc_{label}_{tag}/report.json')
            if not path.exists():
                continue
            m = json.load(open(path, encoding='utf-8'))['metrics']
            d = m['detection']
            neg = d['true_negative'] / max(d['negative_count'], 1)
            print(f"{tag:15s} {label:12s} {neg:10.4f} {d['f1']:8.4f}",
                  f"{m['end_to_end']['overall_success']:8.4f}")


## Phase 7 — Stress test: latency có phẳng theo N không?

Đây là luận điểm bán hàng của Method 2 (§10.2 `experimental_plan.md`):
SLM và API phải nhét cả N tool vào context nên latency tăng tuyến tính,
còn Method 2 chỉ thêm một phép nhân `1×d · d×N` ở Bi-Encoder — Cross-Encoder
không đổi vì chỉ chạy trên tool ĐÃ chọn.

Chạy sau cùng vì dùng lại index 4.464 tool đã dựng ở trên; tách ra notebook
riêng thì phải dựng lại đúng index đó thêm một lần nữa.

Đọc `t_cross_encode` (thành phần chi phối) và `total`. `t_retrieve` ở đây là
**chặn trên**: nó gom hàng từ index dùng chung 4.464 tool bằng một vòng lặp
Python theo tên, thứ mà hệ thống thật — index đúng N tool — không phải làm.


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'src.models.pipeline.stress',
     '--config', 'configs/method2/stress.yaml',
     '--pipeline-config', 'configs/method2/pipeline.yaml'],
    check=True,
)
print(open('results/method2/stress/stress_summary.md', encoding='utf-8').read())


In [ ]:
# Hai biểu đồ §Phase 7: accuracy-vs-N và latency-vs-N, trục x log.
from IPython.display import Image, display

subprocess.run(
    [sys.executable, 'scripts/method2/plot_stress.py',
     '--report', 'results/method2/stress/stress_report.json'],
    check=True,
)
for png in sorted(Path('results/method2/stress').glob('stress_*.png')):
    display(Image(filename=str(png)))


In [ ]:
# ===== Lưu artifact =====
# Kaggle giữ /kaggle/working trong Output của version; tar chỉ là tiện lợi.
# `tar` báo lỗi nếu truyền đường dẫn không tồn tại, nên lọc trước — mỗi
# notebook sinh ra một tập thư mục khác nhau.
_want = ['artifacts/method2', 'results/method2', 'results/evaluation']
_have = [p for p in _want if (Path('/kaggle/working') / p).exists()]
print('đóng gói:', _have)
_paths = ' '.join(_have)
!tar czf /kaggle/working/eval_run.tar.gz -C /kaggle/working {_paths}
!du -h /kaggle/working/eval_run.tar.gz
